# Colab Multitask Soft-Routed Two-Encoder MoE

Notebook này là bản MoE gần hơn với ý tưởng bạn vừa chốt:
- router nhìn `question`
- router ra `softmax([w1, w2])`
- ảnh đi qua **2 visual encoders thật sự**
- lấy `w1 * h1 + w2 * h2`
- dùng một decoder chung để sinh đáp án

Ở đây tôi dùng:
- encoder 1: `ViT`
- encoder 2: `Swin`

Lý do là cặp này giữ được đúng ý tưởng `router-weighted two encoders`, nhưng vẫn dễ train/debug hơn nhiều so với việc ghép thẳng `Pix2Struct + Donut` trong một notebook đầu tiên.


In [ ]:
!python -V
!nvidia-smi


In [ ]:
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio transformers datasets pillow sentencepiece accelerate


In [ ]:
import os
from pathlib import Path

import torch
from datasets import concatenate_datasets, load_dataset
from torch import nn
from transformers import (
    AutoImageProcessor,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    ViTModel,
    SwinModel,
    default_data_collator,
)

WORKDIR = Path('/content/moe_multitask_soft_router')
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
print('Working directory:', WORKDIR)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


In [ ]:
# Change only these config values.
VIT_NAME = 'google/vit-base-patch16-224-in21k'
SWIN_NAME = 'microsoft/swin-tiny-patch4-window7-224'
TEXT_MODEL_NAME = 'google/flan-t5-base'

DOCVQA_SAMPLE_SIZE = 1200
CHARTQA_SAMPLE_SIZE = 1200
VAL_SIZE = 0.1
SEED = 42

BATCH_SIZE = 2
LEARNING_RATE = 1e-4
NUM_EPOCHS = 2
WEIGHT_DECAY = 0.01

FREEZE_VISUAL_ENCODERS = False
VISUAL_HIDDEN_DIM = 512
ROUTER_HIDDEN_DIM = 256
ROUTER_AUX_LOSS_WEIGHT = 0.01

IMAGE_SIZE = 224
MAX_QUESTION_LENGTH = 64
MAX_ANSWER_LENGTH = 48

HF_TOKEN = ''  # optional
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN

print('VIT_NAME =', VIT_NAME)
print('SWIN_NAME =', SWIN_NAME)
print('TEXT_MODEL_NAME =', TEXT_MODEL_NAME)
print('FREEZE_VISUAL_ENCODERS =', FREEZE_VISUAL_ENCODERS)


In [ ]:
def normalize_text(text):
    if text is None:
        return ''
    return ' '.join(str(text).strip().split())


def first_answer(value):
    if isinstance(value, list):
        return value[0] if value else ''
    return value if value is not None else ''


def build_labels(token_ids, pad_token_id):
    return [[-100 if token_id == pad_token_id else token_id for token_id in row] for row in token_ids]


def sample_dataset(dataset, sample_size, seed):
    sample_size = min(sample_size, len(dataset))
    return dataset.shuffle(seed=seed).select(range(sample_size))


## Download and normalize DocVQA + ChartQA

In [ ]:
doc_ds = load_dataset('lmms-lab/DocVQA', 'DocVQA')
doc_split = doc_ds['train'] if 'train' in doc_ds else doc_ds[next(iter(doc_ds.keys()))]
doc_split = sample_dataset(doc_split, DOCVQA_SAMPLE_SIZE, SEED)
doc_dataset = doc_split.map(
    lambda ex: {
        'task': 'docvqa',
        'image': ex['image'],
        'question': normalize_text(ex['question']),
        'answer': normalize_text(first_answer(ex['answers'])),
    },
    remove_columns=doc_split.column_names,
)

chart_ds = load_dataset('HuggingFaceM4/ChartQA')
chart_split = chart_ds['train'] if 'train' in chart_ds else chart_ds[next(iter(chart_ds.keys()))]
chart_split = sample_dataset(chart_split, CHARTQA_SAMPLE_SIZE, SEED)
chart_dataset = chart_split.map(
    lambda ex: {
        'task': 'chartqa',
        'image': ex['image'],
        'question': normalize_text(ex['query']),
        'answer': normalize_text(first_answer(ex['label'])),
    },
    remove_columns=chart_split.column_names,
)

task_dataset = concatenate_datasets([doc_dataset, chart_dataset]).shuffle(seed=SEED)
print(task_dataset)
task_dataset.select(range(5)).to_pandas()


In [ ]:
dataset = task_dataset.train_test_split(test_size=VAL_SIZE, seed=SEED)
print(dataset)


## Preprocess image and text inputs

In [ ]:
vit_processor = AutoImageProcessor.from_pretrained(VIT_NAME)
swin_processor = AutoImageProcessor.from_pretrained(SWIN_NAME)
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)

def preprocess_batch(batch):
    prompted_questions = [
        f'[{task}] question: {normalize_text(question)}'
        for task, question in zip(batch['task'], batch['question'])
    ]
    answers = [normalize_text(answer) for answer in batch['answer']]

    vit_inputs = vit_processor(images=batch['image'], return_tensors='pt')
    swin_inputs = swin_processor(images=batch['image'], return_tensors='pt')

    question_tokens = tokenizer(
        prompted_questions,
        padding='max_length',
        truncation=True,
        max_length=MAX_QUESTION_LENGTH,
    )

    answer_tokens = tokenizer(
        answers,
        padding='max_length',
        truncation=True,
        max_length=MAX_ANSWER_LENGTH,
    )

    return {
        'pixel_values_vit': vit_inputs['pixel_values'],
        'pixel_values_swin': swin_inputs['pixel_values'],
        'router_input_ids': question_tokens['input_ids'],
        'router_attention_mask': question_tokens['attention_mask'],
        'labels': build_labels(answer_tokens['input_ids'], tokenizer.pad_token_id),
        'task_id': [0 if task == 'docvqa' else 1 for task in batch['task']],
    }

MAP_BATCH_SIZE = 8

train_dataset = dataset['train'].map(
    preprocess_batch,
    batched=True,
    batch_size=MAP_BATCH_SIZE,
    writer_batch_size=MAP_BATCH_SIZE,
    remove_columns=dataset['train'].column_names,
)

val_dataset = dataset['test'].map(
    preprocess_batch,
    batched=True,
    batch_size=MAP_BATCH_SIZE,
    writer_batch_size=MAP_BATCH_SIZE,
    remove_columns=dataset['test'].column_names,
)

train_dataset.set_format(type='torch')
val_dataset.set_format(type='torch')

print(train_dataset)
print(val_dataset)


## Define the soft-routed two-encoder model

In [ ]:
class SoftRoutedTwoEncoderModel(nn.Module):
    def __init__(self, vit_name, swin_name, vocab_size, visual_hidden_dim, router_hidden_dim, aux_loss_weight=0.01, freeze_visual_encoders=False):
        super().__init__()
        self.aux_loss_weight = aux_loss_weight

        self.vit = ViTModel.from_pretrained(vit_name)
        self.swin = SwinModel.from_pretrained(swin_name)

        if freeze_visual_encoders:
            for param in self.vit.parameters():
                param.requires_grad = False
            for param in self.swin.parameters():
                param.requires_grad = False

        self.vit_proj = nn.Linear(self.vit.config.hidden_size, visual_hidden_dim)
        self.swin_proj = nn.Linear(self.swin.config.hidden_size, visual_hidden_dim)

        self.router_embedding = nn.Embedding(vocab_size, router_hidden_dim)
        self.router = nn.Linear(router_hidden_dim, 2)

        self.answer_embedding = nn.Embedding(vocab_size, visual_hidden_dim)
        self.decoder = nn.GRU(
            input_size=visual_hidden_dim,
            hidden_size=visual_hidden_dim,
            num_layers=1,
            batch_first=True,
        )
        self.lm_head = nn.Linear(visual_hidden_dim, vocab_size)
        self.loss_fct = nn.CrossEntropyLoss(ignore_index=-100)

    def forward(self, pixel_values_vit, pixel_values_swin, router_input_ids, router_attention_mask, labels=None, task_id=None):
        vit_outputs = self.vit(pixel_values=pixel_values_vit)
        swin_outputs = self.swin(pixel_values=pixel_values_swin)

        vit_hidden = self.vit_proj(vit_outputs.last_hidden_state[:, 0])
        swin_hidden = self.swin_proj(swin_outputs.last_hidden_state.mean(dim=1))

        router_embeds = self.router_embedding(router_input_ids)
        router_mask = router_attention_mask.unsqueeze(-1).float()
        pooled_question = (router_embeds * router_mask).sum(dim=1) / router_mask.sum(dim=1).clamp(min=1.0)
        router_logits = self.router(pooled_question)
        router_weights = torch.softmax(router_logits, dim=-1)

        fused_visual = (
            router_weights[:, 0:1] * vit_hidden +
            router_weights[:, 1:2] * swin_hidden
        )

        if labels is None:
            raise ValueError('labels are required for training in this notebook')

        decoder_input_ids = labels.clone()
        decoder_input_ids[decoder_input_ids == -100] = 0
        decoder_input_ids = torch.roll(decoder_input_ids, shifts=1, dims=1)
        decoder_input_ids[:, 0] = 0

        decoder_embeds = self.answer_embedding(decoder_input_ids)
        decoder_outputs, _ = self.decoder(decoder_embeds, fused_visual.unsqueeze(0))
        logits = self.lm_head(decoder_outputs)

        loss = self.loss_fct(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        mean_usage = router_weights.mean(dim=0)
        target_usage = torch.full_like(mean_usage, 0.5)
        balance_loss = ((mean_usage - target_usage) ** 2).mean()
        total_loss = loss + self.aux_loss_weight * balance_loss

        return {
            'loss': total_loss,
            'logits': logits,
            'router_weights': router_weights.detach(),
            'balance_loss': balance_loss.detach(),
        }


model = SoftRoutedTwoEncoderModel(
    vit_name=VIT_NAME,
    swin_name=SWIN_NAME,
    vocab_size=tokenizer.vocab_size,
    visual_hidden_dim=VISUAL_HIDDEN_DIM,
    router_hidden_dim=ROUTER_HIDDEN_DIM,
    aux_loss_weight=ROUTER_AUX_LOSS_WEIGHT,
    freeze_visual_encoders=FREEZE_VISUAL_ENCODERS,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print('trainable params =', trainable_params)
print('total params =', total_params)


## Train the MoE model

In [ ]:
OUTPUT_DIR = WORKDIR / 'soft_router_two_encoder_outputs'

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=NUM_EPOCHS,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    logging_steps=50,
    report_to='none',
    remove_unused_columns=False,
    seed=SEED,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=default_data_collator,
)

trainer.train()
metrics = trainer.evaluate()
metrics


## Inspect routing behavior

In [ ]:
from collections import defaultdict
from torch.utils.data import DataLoader

model.eval()
model.to(DEVICE)

loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=default_data_collator)
task_name = {0: 'docvqa', 1: 'chartqa'}
usage = defaultdict(list)

with torch.no_grad():
    for step, batch in enumerate(loader):
        if step >= 20:
            break
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        usage[task_name[int(batch['task_id'].item())]].append(outputs['router_weights'].squeeze(0).cpu())

for task, weights in usage.items():
    mean_weight = torch.stack(weights).mean(dim=0)
    print(task, mean_weight)
